In [4]:
import papermill as pm
import duckdb, subprocess, os, time
from joblib import Parallel, delayed
con = duckdb.connect()
con.install_extension("spatial")
con.load_extension("spatial")
con.install_extension("azure")
con.load_extension("azure")
start_time = time.time()
print(time.ctime(time.time()))

Tue Nov  4 13:36:26 2025


In [5]:
# setup parameters for sensitivity analysils
'''
static run

migration waterfowl curves - flatten the curves a few different ways. Set curves static as 1 to 0.5 in increments of 0.1
Do this for all species.  With and without geese.

Goose reduction percentage - 0, 25, 50, 75, 1

habitat availability - Make it all available the entire time[100].  Available early [100,50,25] in the season vs 
mid [33,100,33], vs late [25,50,100].

removewater = True/False

remove moist soil

Reduce/Increase kcalperduck by 10/20%
'''
listfips = {'static':[], 'goosereductionpct':[0,25,75,100], 'removewater':[], 'moistsoil':[],'kcalperduck':[236, 265.5, 324.5, 354], 'energyearly':[100,50,25], 
            'energymid':[33,100,33], 'energylate':[25,50,100]}

In [6]:
# Define function for running
def f(x):
    print(x)
    os.makedirs('./output/', exist_ok=True)
    os.makedirs('./output/plots/', exist_ok=True)
    try:
        if os.path.isfile('./output/'+x+'.parquet'):
            return (x, 'exists')
        else:
            if x == 'static':
                subprocess.run(pm.execute_notebook('Waterfowlmodel-papermill.ipynb','./output/{0}.ipynb'.format(x), shell=True))
                return (x, 'run complete')
            elif x == 'goosereductionpct':
                for pct in listfips[x]:
                    if pct == 0:
                        subprocess.run(pm.execute_notebook('Waterfowlmodel-papermill.ipynb','./output/{0}_{1}.ipynb'.format(x, pct),parameters=dict(name=x+'_'+str(pct), geese=False)), shell=True)
                    else:
                        subprocess.run(pm.execute_notebook('Waterfowlmodel-papermill.ipynb','./output/{0}_{1}.ipynb'.format(x, pct),parameters=dict(name=x+'_'+str(pct), goosereductionpct=pct)), shell=True)
            elif x == 'removewater':
                subprocess.run(pm.execute_notebook('Waterfowlmodel-papermill.ipynb','./output/{0}.ipynb'.format(x),parameters=dict(name=x, removewater=True)), shell=True)
            elif x == 'moistsoil':
                subprocess.run(pm.execute_notebook('Waterfowlmodel-papermill.ipynb','./output/{0}.ipynb'.format(x),parameters=dict(name=x, removemoistsoil=True)), shell=True)
            elif x == 'kcalperduck':
                for pct in listfips[x]:
                    subprocess.run(pm.execute_notebook('Waterfowlmodel-papermill.ipynb','./output/{0}_{1}.ipynb'.format(x, pct),parameters=dict(name=x+'_'+str(pct), kcalperduck=pct)), shell=True)
            elif x == 'energyearly':
                subprocess.run(pm.execute_notebook('Waterfowlmodel-papermill.ipynb','./output/{0}.ipynb'.format(x),parameters=dict(name=x, setcrops=True, setcropavailability=x)), shell=True)
            elif x == 'energymid':
                subprocess.run(pm.execute_notebook('Waterfowlmodel-papermill.ipynb','./output/{0}.ipynb'.format(x),parameters=dict(name=x, setcrops=True, setcropavailability=x)), shell=True)   
            elif x == 'energylate':
                subprocess.run(pm.execute_notebook('Waterfowlmodel-papermill.ipynb','./output/{0}.ipynb'.format(x),parameters=dict(name=x, setcrops=True, setcropavailability=x)), shell=True)                   
    except Exception as e:
        if os.path.isfile('./output/'+x+'.parquet'):
            return (x, 'run complete')
        return (x,'fail', e)

In [ ]:
# Run analysis
completed = Parallel(n_jobs=4, verbose=10)(delayed(f)(key) for key in listfips)

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.


In [33]:
completed

[('static',
  'fail',
  FileNotFoundError(2,
                    'The system cannot find the file specified',
                    None,
                    2,
                    None)),
 None,
 None]